# Sequence-to-sequence, measured: the bottleneck, and attention as the fix

*Companion notebook to the [Sequence-to-Sequence & Encoder–Decoder](../08-Sequence-to-Sequence-and-Encoder-Decoder.md) concept page.*

Hand someone a French sentence and ask for the English: they can't go word-for-word — the lengths differ, the order shuffles, one word becomes three. They **read the whole thing, build a mental representation, then write the translation one word at a time.** That is **sequence-to-sequence**, and the **encoder–decoder** is the architecture that does it.

The vanilla design funnels the entire source through **one fixed context vector** — and that vector is a bottleneck that **collapses with length**. **Attention** removes it by letting the decoder read *all* the encoder states afresh each step. This notebook *measures* both claims from scratch, on the simplest possible seq2seq task — **copying a digit string** (output = input) — so any failure is purely about capacity to carry the source, not reordering or vocabulary.

Every function is imported from the chapter's single source of truth, [`seq2seq.py`](seq2seq.py); the figure generator [`make_figures_08.py`](make_figures_08.py) imports the *same* functions, so the notebook, the page, and the figures cannot drift apart. We will:

1. set up a device-agnostic, seeded run (honest CPU pin);
2. look at the data (the copy task) and one forward pass;
3. train the **bottleneck** and **attention** models (identical except attention);
4. measure the **collapse**: exact-match accuracy vs length, with vs without attention;
5. read the **alignment matrix** — attention as soft, unsupervised word-alignment;
6. compute one **attention step by hand** and check the page's numbers;
7. see **greedy vs beam search** on a worked example.

## Step 1 — device-agnostic, seeded setup (honest CPU pin)

`DEVICE` auto-detects the best accelerator (CUDA → MPS → CPU) so the code runs anywhere, and every tensor and module is moved onto it. For the **published numbers** we pin execution to **CPU**: RNG streams differ between CPU and MPS/CUDA, so a fixed device is what makes "run it and get these numbers" true. The qualitative result — attention ≫ bottleneck — holds on any device.

In [1]:
import numpy as np
import torch

import seq2seq as s2s

RUN_DEVICE = "cpu"  # pin the published table to CPU for reproducibility
s2s.set_seed(0)
print(f"detected device (DEVICE): {s2s.DEVICE}   |   running this notebook on: {RUN_DEVICE}")
print("torch:", torch.__version__, " numpy:", np.__version__)
print("vocab:", s2s.VOCAB_SIZE, "(digits 0-9 + PAD + BOS + EOS)", "| hidden H =", s2s.HIDDEN)

detected device (DEVICE): mps   |   running this notebook on: cpu
torch: 2.12.0  numpy: 2.4.6
vocab: 13 (digits 0-9 + PAD + BOS + EOS) | hidden H = 128


## Step 2 — the data: a copy task, and what one batch looks like

The source is a random digit string ending in `EOS`; the target is the same digits wrapped in `BOS ... EOS` (the decoder learns to start at `BOS` and stop at `EOS`). Copy is the *easiest* seq2seq task — no reordering, no vocabulary change — so the only thing the model can fail at is **carrying the source through the context channel**. That is exactly what isolates the bottleneck. We assert the target really is the BOS-wrapped source before trusting the batch.

In [2]:
src, tgt = s2s.make_batch(batch_size=3, min_len=4, max_len=6, device=RUN_DEVICE)
# Contract: for each row, tgt is BOS, then the source digits (up to EOS), then EOS.
for b in range(src.shape[0]):
    digits = src[b][src[b] < s2s.N_DIGITS].tolist()          # the content digits in the source
    tgt_digits = tgt[b][1 : 1 + len(digits)].tolist()        # target after BOS, before EOS
    assert tgt_digits == digits, "target must be the BOS-wrapped copy of the source"
    assert tgt[b, 0].item() == s2s.BOS and s2s.EOS in tgt[b].tolist()
print("src[0]:", src[0].tolist())
print("tgt[0]:", tgt[0].tolist(), "  (BOS=11, EOS=12, PAD=10)")
print("shapes  src:", tuple(src.shape), " tgt:", tuple(tgt.shape))

src[0]: [3, 3, 7, 9, 12, 10]
tgt[0]: [11, 3, 3, 7, 9, 12, 10]   (BOS=11, EOS=12, PAD=10)
shapes  src: (3, 6)  tgt: (3, 7)


## Step 3 — trace one forward pass: where the bottleneck lives

The encoder reads the source into **both** a full sequence of states `(B, S, H)` *and* a single context vector `(1, B, H)`. The no-attention decoder gets **only** the single vector; the attention decoder reads the whole `(B, S, H)`. Watch the shapes: the `(1, B, H)` summary is the **only** bridge the bottleneck decoder has — all `S` source positions, compressed to one.

In [3]:
enc = s2s.Encoder().to(RUN_DEVICE)
states, context = enc(src)
assert states.shape == (src.shape[0], src.shape[1], s2s.HIDDEN)
assert context.shape == (1, src.shape[0], s2s.HIDDEN)
print("encoder states (B, S, H):", tuple(states.shape), "  <- attention reads ALL of these")
print("context vector (1, B, H):", tuple(context.shape), "  <- the bottleneck decoder gets ONLY this")
print(f"\nthe bottleneck in one line: {src.shape[1]} source positions -> 1 vector of {s2s.HIDDEN} numbers")

encoder states (B, S, H): (3, 6, 128)   <- attention reads ALL of these
context vector (1, B, H): (1, 3, 128)   <- the bottleneck decoder gets ONLY this

the bottleneck in one line: 6 source positions -> 1 vector of 128 numbers


## Step 4 — train both models (identical except for attention)

Same encoder, same size, same steps, same optimizer — the **only** difference is the decoder's attention. That makes this a clean controlled experiment: any gap in the results is attention, nothing else. Training uses **teacher forcing** (feed the gold previous token), which lets all target steps train in one pass and keeps gradients stable. ~4000 steps runs in well under a minute on CPU.

In [4]:
no_attn = s2s.train_model(attention=False, device=RUN_DEVICE)
attn = s2s.train_model(attention=True, device=RUN_DEVICE)
print("trained:", no_attn.tag, "|", attn.tag)

trained: no attention (1 vector) | Bahdanau attention


## Step 5 — the headline: attention holds, the single vector collapses

Free-running **exact-match** accuracy (the decoder feeds on its *own* previous output, exactly as at inference): a sample counts only if the **whole** string is reproduced. We assert the qualitative contract — attention must beat the bottleneck by a wide margin — *before* printing the table.

In [5]:
short_no = s2s.exact_match_accuracy(no_attn, 6, seed=7, device=RUN_DEVICE)
long_no = s2s.exact_match_accuracy(no_attn, 18, seed=8, device=RUN_DEVICE)
short_attn = s2s.exact_match_accuracy(attn, 6, seed=7, device=RUN_DEVICE)
long_attn = s2s.exact_match_accuracy(attn, 18, seed=8, device=RUN_DEVICE)

# Contract: attention >> bottleneck. A single fixed vector simply cannot copy a 6-digit string.
assert short_attn > short_no + 0.3, "attention should crush the bottleneck on copy"

print(f"{'model':>23} | {'short (L=6)':>11} | {'long (L=18)':>11}")
print('-' * 53)
print(f"{no_attn.tag:>23} | {short_no*100:9.1f}% | {long_no*100:9.1f}%")
print(f"{attn.tag:>23} | {short_attn*100:9.1f}% | {long_attn*100:9.1f}%")

                  model | short (L=6) | long (L=18)
-----------------------------------------------------
no attention (1 vector) |       0.7% |       0.0%
     Bahdanau attention |      97.3% |      95.0%


## Step 6 — the bottleneck curve: accuracy vs length

Sweep the source length and watch the two models diverge. The no-attention model collapses almost immediately (it cannot reconstruct a long string from one vector); the attention model stays flat across the whole training range. This is the bottleneck, measured — the same curve [Bahdanau et al. (2015)](https://arxiv.org/abs/1409.0473) showed for BLEU-vs-length on real translation, reproduced here on the simplest task imaginable.

In [6]:
lengths = (2, 4, 6, 8, 10, 12, 16)
acc_no = s2s.accuracy_vs_length(no_attn, lengths, device=RUN_DEVICE)
acc_attn = s2s.accuracy_vs_length(attn, lengths, device=RUN_DEVICE)

# Contract: attention degrades less with length than the bottleneck.
assert acc_attn[-1] > acc_no[-1], "attention should hold up better at the longest length"

print(f"{'length':>7} | {'no-attn':>8} | {'attention':>9}")
print('-' * 32)
for L, a_no, a_attn in zip(lengths, acc_no, acc_attn):
    print(f"{L:>7} | {a_no*100:6.1f}% | {a_attn*100:7.1f}%")

 length |  no-attn | attention
--------------------------------
      2 |   16.3% |   100.0%
      4 |    0.7% |    99.3%
      6 |    0.3% |   100.0%
      8 |    0.3% |    97.7%
     10 |    0.0% |    97.7%
     12 |    1.0% |    99.3%
     16 |    0.7% |    99.7%


The committed figure **`s2s_bottleneck.png`** (generated by `make_figures_08.py` from these same models) plots this divergence — attention flat near 100%, the single vector falling off a cliff:

![Measured exact-match accuracy vs source length: attention holds, the single context vector collapses.](../../images/s2s_bottleneck.png)

## Step 7 — the alignment matrix: attention as soft word-alignment

The most beautiful by-product of attention: the weights $\alpha_{tj}$ form a **soft alignment** between target and source. We never supervised it — the model discovered which source position each output token depends on, purely from the copy objective. On this monotonic task the alignment is a clean **diagonal band** (output token $t$ reads source position $\approx t$); on a *translation* task it would bend and cross wherever the languages reorder. We assert the diagonal band carries most of the mass before reading it.

In [7]:
matrix = s2s.alignment_matrix(attn, [3, 1, 4, 1, 5, 9, 2], device=RUN_DEVICE)
band = s2s.diagonal_band_strength(matrix, band=1)

# Contract: a clean copy alignment concentrates its mass on the diagonal band.
assert band > 0.8, "copy-task attention should concentrate on the diagonal band"

np.set_printoptions(precision=2, suppress=True)
print("alignment matrix (rows = target step, cols = source position):")
print(matrix)
print(f"\nbrightest source per target step: {matrix.argmax(1)}  (a clean diagonal staircase)")
print(f"diagonal-band mass (+/-1): {band:.2f}  (near 1.0 = a sharp copy alignment)")

alignment matrix (rows = target step, cols = source position):
[[0.22 0.17 0.12 0.12 0.11 0.1  0.09 0.07]
 [0.48 0.47 0.02 0.01 0.01 0.01 0.01 0.  ]
 [0.06 0.88 0.05 0.   0.   0.   0.   0.  ]
 [0.   0.01 0.91 0.08 0.   0.   0.   0.  ]
 [0.   0.   0.   0.89 0.1  0.   0.   0.  ]
 [0.   0.   0.   0.01 0.93 0.05 0.   0.  ]
 [0.   0.   0.   0.   0.   0.78 0.2  0.02]
 [0.   0.   0.   0.   0.   0.01 0.62 0.37]]

brightest source per target step: [0 0 1 2 3 4 5 6]  (a clean diagonal staircase)
diagonal-band mass (+/-1): 0.91  (near 1.0 = a sharp copy alignment)


The committed heatmap **`s2s_alignment.png`** shows this as a bright diagonal — the copy rule, learned from scratch:

![Measured attention alignment heatmap: a bright diagonal band = output token t reads source position ~t.](../../images/s2s_alignment.png)

## Step 8 — one attention step, by hand (and it matches the page)

Strip attention to its arithmetic: three toy encoder states, one decoder query, dot-product scores → softmax → weighted blend. These are the exact numbers in the page's worked example — weights `[0.212, 0.212, 0.576]` and context `[0.788, 0.788]` — computed by the same function the page cites, and asserted here.

In [8]:
hand = s2s.attention_step_by_hand()
print("dot-product scores e_j = s . h_j :", hand['scores'])
print("softmax alignment weights       :", np.round(hand['weights'], 3))
print("blended context c = sum_j a_j h_j:", np.round(hand['context'], 3))

# Contract: these are the page's worked-example numbers, exactly.
assert np.allclose(hand['weights'], [0.212, 0.212, 0.576], atol=2e-3)
assert np.allclose(hand['context'], [0.788, 0.788], atol=2e-3)
print("\nmatches the page's worked example.")

dot-product scores e_j = s . h_j : [1. 1. 2.]
softmax alignment weights       : [0.21 0.21 0.58]
blended context c = sum_j a_j h_j: [0.79 0.79]

matches the page's worked example.


## Step 9 — greedy vs beam search

Decoding turns the per-step distributions into a sequence. **Greedy** takes the argmax each step — locally myopic. **Beam search** keeps the top-$k$ partial sequences and approximately maximizes the *sequence* probability. On this worked-example tree, greedy's tied first choice (`A`) shuts the door on `B`'s far better continuation, and beam finds the more probable sequence. We assert beam wins before printing.

In [9]:
step1 = {"A": 0.5, "B": 0.5}
after = {"A": {"X": 0.4, "Y": 0.6}, "B": {"X": 0.9, "Y": 0.1}}
greedy_seq, greedy_prob, ranked = s2s.beam_search_demo(step1, after, beam_width=2)
best_seq, best_prob = ranked[0]

# Contract: beam recovers a higher-probability sequence than greedy.
assert best_prob > greedy_prob, "beam should beat greedy on sequence probability"

print(f"greedy:    '{greedy_seq}'  (p = {greedy_prob:.2f})")
print(f"beam best: '{best_seq}'  (p = {best_prob:.2f})  <- {best_prob/greedy_prob:.0%} of greedy's prob")
print("\nfull beam ranking:")
for seq, p in ranked:
    print(f"  {seq}  p={p:.2f}")

greedy:    'A Y'  (p = 0.30)
beam best: 'B X'  (p = 0.45)  <- 150% of greedy's prob

full beam ranking:
  B X  p=0.45
  A Y  p=0.30
  A X  p=0.20
  B Y  p=0.05


## Recap

We measured the whole story from scratch:

- the encoder–decoder funnels the source through **one fixed context vector**, and that vector **collapses with length** — ~0% copy accuracy at length 6 without attention;
- **attention** removes the bottleneck by reading *all* encoder states afresh each step — ~97–100% across the whole range;
- the attention weights are a **soft, unsupervised alignment** — a clean diagonal on copy, the copy rule discovered from the objective;
- attention's core arithmetic is a **softmax-weighted blend** of encoder states (the by-hand step matches the page);
- **beam search** beats greedy by maximizing *sequence* probability.

From here, the straight line to the Transformer: **self-attention** replaces the encoder's recurrence, **cross-attention** replaces this Bahdanau attention — and you have T5, BART, and Whisper. See the [concept page](../08-Sequence-to-Sequence-and-Encoder-Decoder.md) for that bridge and the full derivations.